In [ ]:
from pathlib import Path

# We use a slightly modified version of the biotrove_process code from the original biotrove repository
from smartrodent.biotrove_process import (
    GenImgTxtPair,
    GenShuffledChunks,
    GetImages,
    MetadataProcessor,
    load_config,
)
from smartrodent.dataprocessing import ImageFilterCLIP

In [ ]:
basepath = Path("../").resolve()
basepath

In [ ]:
config_path = basepath / "configs" / "data_config_central_europe.json"

In [ ]:
datapath = basepath / "datasets" / "BioTrove" / "BioTrove"

# build the image dataset now

In [ ]:
%load_ext autoreload
%autoreload 2
config_path = basepath / "configs" / "data_config_central_europe.json"
config = load_config(config_path)

In [ ]:
params = config.get('metadata_processor_info', {})


In [ ]:
config_path = basepath / "configs" / "data_config_central_europe.json"
config = load_config(config_path)
params = config.get('metadata_processor_info', {})
speciesnames = params["categories"]
speciesnames

In [ ]:
%load_ext autoreload
%autoreload 2

mp = MetadataProcessor(**params)
mp.process_all_files()

# Step 2: generate shuffled chunks of metadata

filter parquet files first.
TODO: put this into the processing package, after making sure it's needed. 

In [ ]:
from pathlib import Path
import polars as pl

src = Path(config["metadata_processor_info"]["source_folder"])
filtered_dir = Path(config["filter_dir"])
categories = config["metadata_processor_info"]["categories"]
category_type = config["metadata_processor_info"]["category_type"]
filtered_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(src.glob("*.parquet")):
    df = (
        pl.scan_parquet(path)
        .filter(pl.col(category_type).is_in(categories))
        .collect()
    )

    if df.height:
        df.write_parquet(filtered_dir / path.name)

then build chunks

In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)
params = config.get('metadata_filter_and_shuffle_info', {})
params["directory"] = str(filtered_dir)
params

In [ ]:
%load_ext autoreload
%autoreload 2
gen_shuffled_chunks = GenShuffledChunks(**params)
gen_shuffled_chunks.process_files()

# Step 3: Download images


In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)

In [ ]:
params = config.get('image_download_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2

gi = GetImages(**params)
await gi.download_images()

# Step 4: Generate text pairs and create tar files (optional)

In [ ]:
%load_ext autoreload
%autoreload 2
config = load_config(config_path)
params = config.get('img_text_gen_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2
textgen = GenImgTxtPair(**params)
textgen.create_image_text_pairs()

## Step 5: Generate classifier dataset

In [ ]:
import shutil

In [ ]:
img_dir = config["img_dir"]
Path(img_dir).mkdir(exist_ok=True, parents = True)

In [ ]:
params = config.get('img_text_gen_info', {})

for chunk in Path(params["img_folder"]).iterdir():
    if chunk.is_dir():
        jpgs = [c for c in Path(chunk).iterdir() if c.suffix == ".jpg"]
        scnames = {c.stem.split(".")[0]: c for c in Path(chunk).iterdir() if 'scientific_name' in str(c)}

        for jpg in jpgs:
            name = jpg.stem

            # find scientific name file
            scientific_name_file = scnames[name]

            with open(scientific_name_file, mode = "r") as scf:
                scientific_name = scf.read()[0:-1]

            species_img_path = (Path(img_dir) / scientific_name)
            species_img_path.mkdir(parents=True, exist_ok=True)

            print(scientific_name, ", ", jpg.name )
            # copy the jpg to the species_img_path
            shutil.copy2(
                jpg,
                str(species_img_path / Path(jpg).name)
            )



# Filtering using CLIP by OpenAI 

This is mostly based on https://colab.research.google.com/github/openai/clip/blob/master/notebooks/Interacting_with_CLIP.ipynb#scrollTo=0BpdJkdBssk9

In [ ]:
import clip
import numpy as np
import torch


check that you have cuda and stuff

In [ ]:
clip.available_models()

define prompts for clip. goal is to filter out any images of dead animals (e.g., traps, roadkill) b/c in citizen scientist datasets that's a common source of images for rodents in particular, and especially for rats an mice

In [ ]:
healthy_prompt = "a live healthy looking animal in its wild habitat or somewhere in a human dwelling or human environment"
dead_prompt = "a dead animal, roadkill, a skull or bones"
not_rodent_prompt = "not an animal at all"
rodent_prompt = " a mouse, rat or other rodent-like animal"

primary_prompts = [ not_rodent_prompt, rodent_prompt]
healthy_or_dead_prompt = [healthy_prompt, dead_prompt]

## Feed in example images

try stuff out a bit first. 

In [ ]:

from pathlib import Path
import random
from smartrodent import ImageFilterCLIP


In [ ]:
imgfilter = ImageFilterCLIP(model = 'RN50x16', prompts=primary_prompts, id_tol = 0.02)

In [ ]:
path = basepath.parent / "datasets" / "biotrove-central-europe" / "imgs" / "Rattus rattus"

In [ ]:
rng_seed = 32456
sample_size = 10
rng = random.Random(rng_seed)
image_paths = rng.sample(list(path.iterdir()), sample_size)

In [ ]:
original_images, images = imgfilter.preprocess_images(image_paths)

In [ ]:
imgfilter.plot_images(original_images)

In [ ]:
similarity = imgfilter.compute_similarity(images)

In [ ]:
decided_idx, undecided_idx, decided, undecided = imgfilter.filter_similarities(
    similarity
)

In [ ]:
decided

In [ ]:
decided.argmax(dim=0)

In [ ]:
decided_idx

In [ ]:
imgfilter.plot_sim_score(decided, [original_images[i] for i in decided_idx], )

In [ ]:
imgfilter.plot_sim_score(undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
rodent_images = [images[i] for i in decided_idx]
rodent_original_images = [original_images[i] for i in decided_idx]

In [ ]:
rodent_filter = ImageFilterCLIP('RN50x16', prompts = healthy_or_dead_prompt, id_tol =0.02)

In [ ]:
# original_images, images = rodent_filter.preprocess_images(rodent_images)
rodent_similarity = rodent_filter.compute_similarity(rodent_images)


In [ ]:
rodent_decided_idx, rodent_undecided_idx, rodent_decided, rodent_undecided = rodent_filter.filter_similarities(
    rodent_similarity
)

In [ ]:
rodent_filter.plot_sim_score(rodent_decided, [original_images[i] for i in rodent_decided_idx], )

In [ ]:
rodent_decided.argmax(dim=0)

In [ ]:
rodent_filter.plot_sim_score(rodent_undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
labeled_data = rodent_filter.decisions(rodent_original_images, rodent_decided)

In [ ]:
labeled_data

In [ ]:
fig, ax = imgfilter.plot_images(labeled_data['a dead animal, roadkill, a skull or bones'])
fig.show()

In [ ]:
imgfilter.plot_images(labeled_data[healthy_prompt])

## Sequential CLIP filtering for all Central Europe species

This cell walks every species folder under `datasets/biotrove-central-europe/imgs`, first keeps images that CLIP labels as rodent-like, then keeps only the subset also labeled as live/healthy. Kept images are copied into `filtered/` and all rejected, dead, or uncertain images are copied into `filtered_dead_or_unsure/`, preserving the species-folder label structure.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import shutil
from tqdm.auto import tqdm
import torch
from smartrodent.dataprocessing import (
    ImageFilterCLIP,
    ImageFilterYoloE,
    ImageFilterBiotroveClip,
)
tols= [0.01, 0.05, 0.02, 0.1]
# Source directory: one subdirectory per species; each subdirectory name is the label.
imgs_root = Path(
    "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/imgs"
)
results = {}
# First pass: animal/rodent-like versus not-an-animal.
not_rodent_prompt = "not an animal at all"
rodent_prompt = "a mouse, rat or other rodent-like animal"

# Second pass: usable live-animal images versus dead/specimen/uncertain images.
healthy_prompt = "a live healthy looking animal in its wild habitat or somewhere in a human dwelling or human environment"
dead_or_unsure_prompt = "a dead animal, roadkill, a skull or bones"

for tol in tols:

    results[tol] = {}

    primary_filter_clip = ImageFilterCLIP(
        model="RN50x16",
        prompts=[not_rodent_prompt, rodent_prompt],
        id_tol=tol,
    )
    health_filter_clip = ImageFilterCLIP(
        model="RN50x16",
        prompts=[healthy_prompt, dead_or_unsure_prompt],
        id_tol=tol,
    )

    # YOLOE is loaded through Ultralytics from the local weights bundled under
    # src/smartrodent, then configured with the text prompts below via set_classes.

    primary_filter_yoloe = ImageFilterYoloE(
        model="yoloe-26m-seg.pt",
        prompts=[not_rodent_prompt, rodent_prompt],
        id_tol=tol,
        predict_conf=tol,
    )
    health_filter_yoloe = ImageFilterYoloE(
        model="yoloe-26m-seg.pt",
        prompts=[healthy_prompt, dead_or_unsure_prompt],
        id_tol=tol,
        predict_conf=tol,
    )

    biotrove_clip_checkpoint = "biotroveclip-vit-b-16-from-openai-epoch-40.pt"

    primary_filter_biotrove_clip = ImageFilterBiotroveClip(
        model=biotrove_clip_checkpoint,
        prompts=[not_rodent_prompt, rodent_prompt],
        id_tol=tol,
    )
    health_filter_biotrove_clip = ImageFilterBiotroveClip(
        model=biotrove_clip_checkpoint,
        prompts=[healthy_prompt, dead_or_unsure_prompt],
        id_tol=tol,
    )

    for primary_filter, health_filter, name in zip(
        [primary_filter_clip, primary_filter_yoloe, primary_filter_biotrove_clip],
        [health_filter_clip, health_filter_yoloe, health_filter_biotrove_clip],
        ["clip", "yoloe", "biotroveclip"],
    ):
        filtered_root = imgs_root.parent / f"filtered_{name}_{tol}"
        rejected_root = imgs_root.parent / f"filtered_{name}_rejected_{tol}"
        undecided_root = imgs_root.parent / f"filtered_{name}_undecided_{tol}"

        # Leave this False to avoid deleting previous results accidentally. Set to True
        # when rebuilding the filtered directories from scratch.
        clear_existing_outputs = False

        if clear_existing_outputs:
            shutil.rmtree(filtered_root, ignore_errors=True)
            shutil.rmtree(rejected_root, ignore_errors=True)
            shutil.rmtree(undecided_root, ignore_errors=True)

        filtered_root.mkdir(parents=True, exist_ok=True)
        rejected_root.mkdir(parents=True, exist_ok=True)
        undecided_root.mkdir(parents=True, exist_ok=True)

        image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

        def copy_with_structure(src: Path, dst_root: Path) -> Path:
            """Copy one image to dst_root while preserving its path below imgs_root."""
            relative_path = src.relative_to(imgs_root)
            dst = dst_root / relative_path
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            return dst

        def decided_label(
            image_filter: ImageFilterCLIP, preprocessed_images: list
        ) -> tuple[str | None, torch.Tensor]:
            """Return the winning prompt only when CLIP made a non-ambiguous decision."""
            similarity = image_filter.compute_similarity(preprocessed_images)
            decided_idx, _, decided, _ = image_filter.filter_similarities(similarity)
            if len(decided_idx) != 1:
                return None, similarity
            return image_filter.prompts[int(decided.argmax(dim=0).item())], similarity

        species_dirs = sorted(p for p in imgs_root.iterdir() if p.is_dir())
        image_paths = [
            image_path
            for species_dir in species_dirs
            for image_path in sorted(species_dir.rglob("*"))
            if image_path.is_file() and image_path.suffix.lower() in image_suffixes
        ]

        summary = Counter()
        by_species = defaultdict(Counter)
        errors = []

        for image_path in tqdm(image_paths, desc=f"{name} filtering images"):
            species = image_path.relative_to(imgs_root).parts[0]
            try:
                _, preprocessed = primary_filter.preprocess_images([image_path])
                primary_label, _ = decided_label(primary_filter, preprocessed)

                if primary_label is None:
                    copy_with_structure(image_path, undecided_root)
                    reason = "rodent_unsure"
                elif primary_label != rodent_prompt:
                    copy_with_structure(image_path, rejected_root)
                    reason = "not_rodent"
                else:
                    health_label, _ = decided_label(health_filter, preprocessed)
                    if health_label is None:
                        copy_with_structure(image_path, undecided_root)
                        reason = "health_unsure"
                    elif health_label == healthy_prompt:
                        copy_with_structure(image_path, filtered_root)
                        reason = "alive"
                    else:
                        copy_with_structure(image_path, rejected_root)
                        reason = "dead"

                summary[reason] += 1
                by_species[species][reason] += 1
            except Exception as exc:
                copy_with_structure(image_path, undecided_root)
                summary["error"] += 1
                by_species[species]["error"] += 1
                errors.append((str(image_path), repr(exc)))

        print("Input:", imgs_root)
        print("tol  : ", tol)
        print("name : ", name)
        print(" Kept live rodent images:", filtered_root)
        print(" Rejected/not-rodent/dead images:", rejected_root)
        print(" Undecided/error images:", undecided_root)
        print(" Total images:", len(image_paths))
        print(" Summary:", dict(summary))
        print(" By species:")
        for species, counts in sorted(by_species.items()):
            print(species, dict(counts))

        results[tol][name] = by_species

        if errors:
            print(f"Encountered {len(errors)} errors; first 10:")
            for path, error in errors[:10]:
                print(path, error)
        print("$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$")
results

## Try to use locally hosted qwen3.6 model to run filtering

In [ ]:
import json
import requests
from pathlib import Path
import base64
import pandas as pd
from tqdm import tqdm
import shutil

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen3.6:27b"

PROMPT = """
   You are classifying wildlife images.

   Return ONLY valid JSON with this exact schema:

   {
     "label": "alive | dead | unsure",
     "visible_animal": true,
     "evidence_alive": ["string"],
     "evidence_dead": ["string"],
     "image_quality": "clear | poor | unusable",
     "needs_human_review": true
   }

   Rules:
   - label must be exactly one of: "alive", "dead", "unsure".
   - Use "dead" only for clear evidence of death: carcass, roadkill, skull, bones,
 preserved specimen, visibly dead body.
   - Use "alive" only if a live animal is clearly visible.
   - Use "unsure" if no animal is visible, the image is ambiguous, cropped, low
 quality, a drawing, or you cannot confidently decide.
   - needs_human_review should be true for unsure, poor/unusable images, or conflicting
 evidence.
   - Do not include markdown.
   - Do not include text outside the JSON object.
   """

image_suffixes = {".jpg", ".jpeg", ".png"}


def copy_with_structure(src: Path, dst_root: Path) -> Path:
    """Copy one image to dst_root while preserving its path below imgs_root."""
    relative_path = src.relative_to(imgs_root)
    dst = dst_root / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst


def classify_image(path: str, prompt: str) -> dict:
    img_path = Path(path)
    img_b64 = base64.b64encode(img_path.read_bytes()).decode("utf-8")
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "images": [img_b64],
        "format": "json",
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0,
        },
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()

    raw = response.json()["response"]

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return {
            "label": "failure",
            "visible_animal": None,
            "evidence_alive": [],
            "evidence_dead": [],
            "image_quality": "unknown",
            "needs_human_review": True,
            "raw_response": raw,
            "parse_error": True,
        }

    label = str(data.get("label", "failure")).strip().lower()

    if label not in ["alive", "dead", "unsure"]:
        label = "failure"

    return {
        "label": label,
        "visible_animal": data.get("visible_animal"),
        "evidence_alive": data.get("evidence_alive", []),
        "evidence_dead": data.get("evidence_dead", []),
        "image_quality": data.get("image_quality", "unsure"),
        "needs_human_review": bool(data.get("needs_human_review", label == "unsure")),
        "raw_response": raw,
        "parse_error": False,
    }


def filter_data_ollama(
    prompt: str,
    raw_data_path: str,
    alive_path: str,
    unsure_path: str,
    dead_path: str,
    failure_path: str,
) -> pd.DataFrame:

    results = []
    species_dirs = sorted(p for p in Path(raw_data_path).iterdir() if p.is_dir())
    image_paths = [
        image_path
        for species_path in species_dirs
        for image_path in sorted(species_path.iterdir())
        if image_path.is_file() and image_path.suffix.lower() in image_suffixes
    ]

    for image_path in tqdm(image_paths, desc="OLLAMA filtering images"):
        res = classify_image(image_path, prompt)
        res["species"] = image_path.relative_to(raw_data_path).parts[0]
        results.append(res)

        if res["label"] == "alive":
            copy_with_structure(image_path, alive_path)
        elif res["label"] == "dead":
            copy_with_structure(image_path, dead_path)
        elif res["label"] == "unsure":
            copy_with_structure(image_path, unsure_path)
        elif res["label"] == "failure":
            copy_with_structure(image_path, failure_path)
        else:
            raise ValueError(f"Unknown label {res['label']}")

    return pd.DataFrame(results)


imgs_root = Path(
    "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/imgs"
)
alive_root = imgs_root.parent / "filtered_qwen3.6"
unsure_root = imgs_root.parent / "filtered_qwen3.6_undecided"
dead_root = imgs_root.parent / "filtered_qwen3.6_rejected"
failure_root = imgs_root.parent / "filtere_qwen3.6_failure"

res_df = filter_data_ollama(
    PROMPT,
    imgs_root,
    alive_root,
    unsure_root,
    dead_root,
    failure_root
)

res_df.to_csv(imgs_root.parent / "filter_results_qwen36.csv")

## Compare all the results of the filtering

In [ ]:
import pandas as pd
import json

In [ ]:
results_normal = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filter_results_other.csv"
results_qwen36 = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filter_results_qwen36.csv"

In [ ]:
df_normal = pd.read_csv(results_normal)

In [ ]:
df_normal

In [ ]:
df_normal= df_normal.drop(columns=["Unnamed: 0"])


In [ ]:
df_qwen =pd.read_csv(results_qwen36)

In [ ]:
df_qwen

#### harmonize columns

In [ ]:
df_qwen_agg = []
df_qwen =pd.read_csv(results_qwen36)
for species, group in df_qwen.groupby("species"):
    res = {
        "model": "qwen3.6-27b",
        "tolerance": None, # no tolerance for qwen
        "species": species,
        "kept_live_rodent": 0,
        "dead_or_unusable": 0,
        "primary_unsure": 0,
        "health_unsure": 0,
        "not_rodent": 0
    }
    for raw_response in group["raw_response"]:
        response = json.loads(raw_response)
        label = response["label"]
        visible_animal = response["visible_animal"]
        needs_human_review = response["needs_human_review"]
        res["kept_live_rodent"] += int(label == "alive" and visible_animal)
        res["dead_or_unusable"] += int(label == "dead" and visible_animal)
        res["primary_unsure"] += int(label == "unsure" and not visible_animal)
        res["health_unsure"] += int(label == "unsure" and visible_animal)
        res["not_rodent"] += int(label != "unsure" and not visible_animal)

    df_qwen_agg.append(res)

df_qwen = pd.DataFrame(df_qwen_agg)
df_qwen


In [ ]:
df_models = pd.concat([df_normal, df_qwen])
df_models

### visualize model results

In [ ]:
# Flatten and visualize filtering experiment results
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

results=df_models.fillna(-0.1)

count_columns = [
    "kept_live_rodent",
    "dead_or_unusable",
    "not_rodent",
    "primary_unsure",
    "health_unsure",
    "error",
]
for column in count_columns:
    if column not in results.columns:
        results[column] = 0
    results[column] = results[column].astype(int)

plot_columns = [
    "kept_live_rodent",
    "dead_or_unusable",
    "not_rodent",
    "primary_unsure",
    "health_unsure",
]

# Aggregate over species so each bar is one model/tolerance combination.
plot_df = (
    results
    .groupby(["tolerance", "model", "species"], as_index=False)[plot_columns]
    .sum()
    .melt(
        id_vars=["tolerance", "model", "species"],
        value_vars=plot_columns,
        var_name="category",
        value_name="image_count",
    )
)
plot_df["image_count_plot"] = plot_df["image_count"] + 1
sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=plot_df,
    kind="bar",
    x="tolerance",
    y="image_count",
    hue="species",
    col="category",
    row="model",
    sharey=False,
    height=4,
    aspect=1.4,
    log_scale=False,
    margin_titles=True,

)
# g.set_xaxis_labels("Tolerance")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.figure.suptitle("Filtering results by model, tolerance, and decision category", y=1.03)
g.figure.show()

g.figure.savefig("/home/hmack/Development/rodent_experiments/notes/model_filter_comparison.pdf")


### Agreement matrices 

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

tols = [None, 0.01, 0.02, 0.05, 0.1]
models = ["biotroveclip", "clip", "yoloe", "qwen3.6"]
model_labels = []
for model in models:
    for tol in tols:
        if model != "qwen3.6" and tol is None:
            continue
        elif model == "qwen3.6" and tol is not None:
            continue
        else:
            model_labels.append((model, tol))

basepath = Path("/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe").resolve()

decisions = ["kept", "rejected", "unsure"]
image_suffixes = {".jpg", ".jpeg", ".png",}
decision_dirs = {
    "kept": "filtered_{model}{suffix}",
    "rejected": "filtered_{model}_rejected{suffix}",
    "unsure": "filtered_{model}_undecided{suffix}",
}

def suffix_for(model, tol):
    # Qwen was generated once, without tolerance-specific output folders.
    if model == "qwen3.6":
        return ""
    if tol is None:
        return None
    return f"_{tol}"

def source_image_count(sp):
    directory = (basepath / "imgs" / sp).resolve()
    if not directory.exists():
        print("not found: ", directory)
        return 0

    return sum(
        1
        for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in image_suffixes
    )

def image_names(model, tol, decision, sp):
    suffix = suffix_for(model, tol)
    if suffix is None:
        return set()

    dirname = decision_dirs[decision].format(model=model, suffix=suffix)
    directory = basepath / dirname / sp
    if not directory.exists():
        return set()
    return {p.name for p in directory.iterdir() if p.is_file()}

species_image_counts = {sp: source_image_count(sp) for sp in speciesnames}

records = []

for model_label_row in model_labels:
    for model_label_col in model_labels:
        model_row, tol_row = model_label_row
        model_col, tol_col = model_label_col
        for sp in species:
            row = {
                "model_row": f"{model_row}_{tol_row}",
                "model_col": f"{model_col}_{tol_col}",
                "species": sp,
                "species_image_count": species_image_counts[sp],
            }
            denominator = species_image_counts[sp]
            for decision in decisions:
                row_imgs = image_names(model_row, tol_row, decision, sp)
                col_imgs = image_names(model_col, tol_col, decision, sp)
                overlap_count = len(row_imgs & col_imgs)

                row[decision] = overlap_count
                row[f"{decision}_relative"] = overlap_count / (max(len(row_imgs), len(col_imgs)) or np.nan)
            records.append(row)

data = pd.DataFrame.from_records(records).reset_index(drop=True)

data.to_csv(basepath/ "model_filtering_agreement.csv")

In [ ]:
data


In [ ]:
for decision in ["kept_relative", "rejected_relative", "unsure_relative"]:
    fig, axs = plt.subplots(3, 4, figsize=(26,20))
    axs = axs.flatten()
    axi = 0
    for spt, group in data.groupby("species"):
        to_plot = group.pivot(index = "model_row", columns = "model_col", values = f"{decision}")
        sns.heatmap(to_plot, annot=False, ax = axs[axi], vmin=0.0, vmax = 1.0)
        axs[axi].set_title(spt)
        axi+=1

    for i in range(axi, len(axs)):
        axs[i].set_axis_off()

    fig.suptitle(f"Relative overlap of {decision} images per species")
    fig.tight_layout()
    fig.savefig(basepath/ f"relative_overlap_between_models_{decision}.jpg")


## Generate dataset from filtered images and speciesnet results 

In [ ]:
from smartrodent.detection import SpeciesNet_Detector
from pathlib import Path

In [ ]:
dataset_output_path = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/yolo_detector_training_dataset"
path_to_data = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered_final"
path_to_labels = "/home/hmack/Development/rodent_experiments/datasets/training-central-europe/speciesnet_box_labels_nopad"
class_names = speciesnames
image_directory = "boxed"

#### create boxes with speciesnet 

In [ ]:
detector_kwargs={"model_name": "kaggle:google/speciesnet/pyTorch/v4.0.3b/1",
                "crop": True,
                "country": "DEU",
                "relpad": 0.0,
                }

In [ ]:
def image_groups(root: Path):
    """Yield named image batches from immediate subdirectories of ``root``.

    The current dataset layout groups images by folder. Each yielded tuple is
    ``(folder_name, sorted_image_paths)`` and empty/non-directory entries are ignored.
    """
    for imgpath in sorted(root.iterdir()):
        if not imgpath.is_dir():
            continue
        imgs = sorted(imgpath.iterdir())
        if imgs:
            yield imgpath.name, imgs


In [ ]:
path_to_data

In [ ]:
path_to_labels

In [ ]:
detector = SpeciesNet_Detector(
        name="boxed",
        batchsize=80,
        img_outname="boxed",
        conf=0.01,
        project=path_to_labels,
        **detector_kwargs,
    )
for group_name, imgs in image_groups(Path(path_to_data)):
    print(group_name)
    detector.detect(imgs, Path(path_to_labels) / group_name)

#### create boxes with pytorch-wildlife megadetector version (to test)

In [ ]:
from PytorchWildlife.models import detection as pw_detection

In [ ]:
path_to_data = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered_final"
path_to_labels = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/pw_megadetector_box_labels"



In [ ]:
detector = pw_detection.MegaDetectorV6MIT(version="MDV6-mit-yolov9-e")

In [ ]:
detection = detector.single_image_detection("/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered_final/Apodemus agrarius/29433667.jpg")
detection

#### create dataset for yolo detector

In [ ]:
from smartrodent.dataprocessing import YoloDetectorDatasetCreatorFromSpeciesnet

config_path = basepath / "configs" / "data_config_central_europe.json"
config = load_config(config_path)
params = config.get('metadata_processor_info', {})
speciesnames = params["categories"]

dataset_output_path = "/home/hmack/Development/rodent_experiments/datasets/training-central-europe/yolo_detector_training_dataset"
path_to_data = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered_final"
path_to_labels = "/home/hmack/Development/rodent_experiments/datasets/training-central-europe/speciesnet_box_labels_nopad"

image_directory = "boxed"

dataset_creator = YoloDetectorDatasetCreatorFromSpeciesnet(
    path_to_image_data = path_to_data,
    path_to_labels = path_to_labels,
    dataset_output_path = dataset_output_path,
    labels_to_filter = ['animal', 'rodent'],
    class_names = speciesnames,
    train_val_test_split = (0.7, 0.15, 0.15),
    IoU_threshold=0.2,
    confidence_threshold=0.15

)
dataset = dataset_creator()

#### create dataset for yolo classifier

In [ ]:
from smartrodent.dataprocessing import YoloClassifierDatasetCreatorFromSpeciesnet

config_path = basepath / "configs" / "data_config_central_europe.json"
config = load_config(config_path)
params = config.get('metadata_processor_info', {})
speciesnames = params["categories"]

dataset_output_path = "/home/hmack/Development/rodent_experiments/datasets/training-central-europe/yolo_classifier_training_dataset"
path_to_data = "/home/hmack/Development/rodent_experiments/datasets/training-central-europe/speciesnet_box_labels_nopad"
class_names = speciesnames

image_directory = "crops"

dataset_generator = YoloClassifierDatasetCreatorFromSpeciesnet(
    path_to_image_data = path_to_data,
    class_names = class_names,
    dataset_output_path=dataset_output_path,
    labels_to_filter = ['animal', 'rodent'],
    train_val_test_split = (0.7, 0.15, 0.15),
    IoU_threshold=0.2,
    confidence_threshold=0.15
)

dataset = dataset_generator()

#### create dataset for speciesnet/megadetector classifier